# Litetorch GPU/TPU Test — Colab

**Cách dùng:** Colab → *Runtime* → *Change runtime type* → chọn **T4 GPU** (để test GPU) hoặc **TPU** (để test TPU), rồi *Runtime* → *Run all*.

Notebook này clone nhánh `agent`, build từ source và chạy các test kiểm chứng những bug đã fix cần phần cứng thật.

In [ ]:
import os, subprocess, shutil
gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip().split('\n')[0] if shutil.which('nvidia-smi') else 'none'
print('COLAB_GPU =', gpu)
print('COLAB_TPU_ADDR =', os.environ.get('COLAB_TPU_ADDR'))
print('nvcc:', shutil.which('nvcc') or 'missing')


In [ ]:
!pip install -q pybind11
import os
if os.environ.get('COLAB_TPU_ADDR'):
    print('TPU runtime detected: installing libtpu...')
    !pip install -q libtpu
    !python3 -c "import libtpu; print('libtpu ok')" 2>&1 | tail -1

In [ ]:
%cd /content
!rm -rf Litetorch-
!git clone --branch agent --depth 1 https://github.com/nguyenminh20000/Litetorch-.git 2>&1 | tail -2
%cd Litetorch-
!git log --oneline -1

In [ ]:
import shutil, os
if shutil.which('nvcc') is None and os.path.exists('/usr/local/cuda/bin/nvcc'):
    os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
    print('added /usr/local/cuda/bin to PATH')
print('nvcc:', shutil.which('nvcc'))

!LITETORCH_CUDA=1 python3 setup.py build_ext --inplace 2>&1 | tail -4

import glob
gpu_sos = glob.glob('build/lib*/liblitetorch_gpu.so')
print('gpu .so:', gpu_sos)
for s in gpu_sos:
    !cp {s} ./liblitetorch_gpu.so
    print('copied to ./liblitetorch_gpu.so')
    break

In [ ]:
import sys
sys.path.insert(0, '/content/Litetorch-')
import litetorch as lt

print('is_gpu_available:', lt.is_gpu_available())
print('tpu.is_available:', lt.tpu.is_available())

passed, failed = [], []
def check(name, cond):
    (passed if cond else failed).append(name)
    print(('PASS' if cond else 'FAIL'), name)

def expect_throw(name, fn):
    try:
        fn()
    except Exception as e:
        check(name, True); return
    check(name + ' (no throw!)', False)

CPU = lt.Device('cpu')
def T(data, shape, device=CPU, rg=False, dtype=lt.DataType.FP32):
    return lt.Tensor.from_vector(data, shape, device, rg, dtype)

if lt.is_gpu_available():
    GPU = lt.Device('gpu:0')
    Ops = lt.Ops

    # 1. matmul GPU mixed dtype must throw (was silent garbage)
    expect_throw('gpu matmul mixed dtype throws', lambda: Ops.matmul(
        T([1.]*4,[2,2],GPU), T([1.]*4,[2,2],GPU).cast(lt.DataType.FP16)))

    # 2. matmul GPU correctness vs CPU
    a = T([1.,2.,3.,4.],[2,2],GPU); b = T([5.,6.,7.,8.],[2,2],GPU)
    r = Ops.matmul(a,b).to(CPU).to_vector()
    check('gpu matmul correct', r == [19.,22.,43.,50.])

    # 3. conv2d GPU forward smoke + FP16 must throw
    x = T([1.]*16,[1,1,4,4],GPU); w = T([1.]*4,[1,1,2,2],GPU)
    o = Ops.conv2d(x,w).to(CPU).to_vector()
    check('gpu conv2d fwd', o == [4.]*9)
    expect_throw('gpu conv2d FP16 throws', lambda: Ops.conv2d(
        x.cast(lt.DataType.FP16), w.cast(lt.DataType.FP16)))

    # 4. maxpool2d GPU forward/backward with int32 indices
    xp = T([float(i) for i in range(16)],[1,1,4,4],GPU,True)
    o = Ops.max_pool2d(xp,2)
    Ops.sum(o).backward()
    check('gpu maxpool fwd', o.to(CPU).to_vector()==[5.,7.,13.,15.])
    check('gpu maxpool bwd', abs(sum(xp.grad.to(CPU).to_vector())-4.0)<1e-4)

    # 5. flash_attention GPU smoke (tests scratch alloc/free sync fix)
    q = T([0.1]*32,[1,2,4,4],GPU); k = T([0.1]*32,[1,2,4,4],GPU); v = T([0.1]*32,[1,2,4,4],GPU)
    for _ in range(3):
        fa = Ops.flash_attention(q,k,v)
    check('gpu flash_attention runs x3', fa.to(CPU).to_vector()[0] > 0)

    # 6. paged_attention GPU with INT32 tables; float tables must throw
    q = T([0.5]*8,[1,1,2,4],GPU); k = T([0.5]*16,[1,2,2,4],GPU); v = T([0.5]*16,[1,2,2,4],GPU)
    bt = T([0,1],[2],GPU,dtype=lt.DataType.INT32); cl = T([2],[1],GPU,dtype=lt.DataType.INT32)
    po = Ops.paged_attention(q,k,v,bt,cl)
    check('gpu paged_attention INT32', list(po.shape)==[1,1,2,4])
    expect_throw('gpu paged_attention float tables throws',
        lambda: Ops.paged_attention(q,k,v,T([0.,1.],[2],GPU),cl))

    # 7. mse_loss GPU mean (was sum) — matches CPU
    p = T([1.,2.,3.],[3],GPU,True); t = T([1.,1.,1.],[3],GPU)
    loss = Ops.mse_loss(p,t)
    check('gpu mse_loss mean', abs(loss.to(CPU).item() - 5/3) < 1e-4)
else:
    print('No GPU — skipping GPU tests')

print(f'\nGPU section: {len(passed)} passed, {len(failed)} failed')

In [ ]:
tpassed, tfailed = [], []
def tcheck(name, cond):
    (tpassed if cond else tfailed).append(name)
    print(('PASS' if cond else 'FAIL'), name)

if lt.tpu.is_available():
    TPU = lt.Device('tpu:0')
    Ops = lt.Ops

    # 1. TPU matmul FP32 correctness vs CPU (element-vs-byte offset fix)
    a = T([1.,2.,3.,4.],[2,2],TPU); b = T([5.,6.,7.,8.],[2,2],TPU)
    r = Ops.matmul(a,b).to(CPU).to_vector()
    tcheck('tpu matmul FP32 correct', r == [19.,22.,43.,50.])

    # 2. transpose offset on TPU (was byte-offset bug)
    x = T([float(i) for i in range(6)],[2,3],TPU)
    xt = x.transpose(0,1)
    tcheck('tpu transpose offset', xt.to(CPU).to_vector() == [0.,3.,1.,4.,2.,5.])

    # 3. non-FP32 matmul must not crash (fallback cast path)
    try:
        h = T([1.]*4,[2,2],TPU).cast(lt.DataType.FP16)
        ro = Ops.matmul(h,h).to(CPU).to_vector()
        tcheck('tpu FP16 matmul no-crash', True)
    except Exception as e:
        tcheck('tpu FP16 matmul no-crash (threw cleanly: %s)' % type(e).__name__, True)

    # 4. CPU->TPU async copy lands on device
    xc = T([9.,8.,7.],[3],CPU)
    xt2 = xc.to(TPU)
    tcheck('cpu->tpu copy', xt2.to(CPU).to_vector() == [9.,8.,7.])
else:
    print('No TPU — skipping TPU tests')

print(f'\nTPU section: {len(tpassed)} passed, {len(tfailed)} failed')

## Kết quả

- GPU section và TPU section in số PASS/FAIL riêng.
- Nếu có FAIL, copy traceback gửi lại để mình fix tiếp.